### **Adaboost** (Classification)

In [3]:
# ==============================================
# STEP-BY-STEP ADABOOST IMPLEMENTATION (FROM SCRATCH)
# ==============================================

import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

# ==============================================
# CREATE DUMMY DATA
# ==============================================
X, y = make_classification(n_samples=20, n_features=2, n_redundant=0,
                           n_informative=2, random_state=42)       #2D → easy to visualize

# Convert labels from {0,1} → {-1, +1}
y = np.where(y == 0, -1, 1)

n_samples = len(y)

# ==============================================
# STEP 1: INITIALIZE WEIGHTS
# ==============================================
weights = np.ones(n_samples) / n_samples     #an array of all 1’s
print("\nSTEP 1: Initial Weights:")
print(weights)

# PARAMETERS
n_estimators = 5
learning_rate = 1.0

models = []
alphas = []

# ==============================================
# BOOSTING LOOP
# ==============================================
for t in range(n_estimators):
    print(f"\n================ ITERATION {t+1} ================")

    # ==============================================
    # STEP 2: TRAIN WEAK LEARNER
    # ==============================================
    stump = DecisionTreeClassifier(max_depth=1)
    stump.fit(X, y, sample_weight=weights)

    predictions = stump.predict(X)

    print("Predictions:", predictions)

    # Identify misclassified points
    misclassified = (predictions != y)
    print("Misclassified:", misclassified)

    # ==============================================
    # STEP 3: COMPUTE ERROR
    # ==============================================
    error = np.sum(weights * misclassified)
    print("Error:", error)

    # Avoid divide by zero
    error = max(error, 1e-10)

    # ==============================================
    # STEP 4: COMPUTE ALPHA (MODEL WEIGHT)
    # ==============================================
    alpha = learning_rate * 0.5 * np.log((1 - error) / error)
    print("Alpha (Model Weight):", alpha)

    # ==============================================
    # STEP 5: UPDATE WEIGHTS
    # ==============================================
    # Increase weight for misclassified, decrease for correct
    weights = weights * np.exp(-alpha * y * predictions)

    # Normalize weights
    weights = weights / np.sum(weights)

    print("Updated Weights:", weights)

    # Store model and alpha
    models.append(stump)
    alphas.append(alpha)

# ==============================================
# STEP 7: FINAL PREDICTION
# ==============================================
print("\n===== FINAL PREDICTION =====")
#Combine all weak learners (stumps) into one final prediction
def adaboost_predict(X, models, alphas):
    final = np.zeros(X.shape[0])   #final = [0, 0, 0, 0, 0], store combined result
#zip() pairs them: (0.8, stump1)
    for alpha, model in zip(alphas, models):
        pred = model.predict(X)
        final += alpha * pred

    return np.sign(final)

final_predictions = adaboost_predict(X, models, alphas)

print("Final Predictions:", final_predictions)
print("Actual Labels:", y)

#X → data
#models → list of trained stumps
#alphas → importance of each stump



STEP 1: Initial Weights:
[0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05]

================ ITERATION 1 ================
Predictions: [-1  1  1  1  1 -1  1  1  1  1 -1  1  1 -1 -1  1 -1 -1  1 -1]
Misclassified: [False  True  True False False False False False  True False False False
 False False False False False False False False]
Error: 0.15000000000000002
Alpha (Model Weight): 0.8673005276940532
Updated Weights: [0.02941176 0.16666667 0.16666667 0.02941176 0.02941176 0.02941176
 0.02941176 0.02941176 0.16666667 0.02941176 0.02941176 0.02941176
 0.02941176 0.02941176 0.02941176 0.02941176 0.02941176 0.02941176
 0.02941176 0.02941176]

================ ITERATION 2 ================
Predictions: [-1 -1 -1  1  1 -1  1 -1  1  1 -1  1  1 -1 -1  1 -1 -1 -1 -1]
Misclassified: [False False False False False False False  True  True False False False
 False False False False False False  True False]
Error: 0.2254901960784314
Alpha (Model We

**Adaboost (Regression)**

In [14]:
# ==============================================
# ADABOOST FOR REGRESSION (STEP-BY-STEP)
# ==============================================

import numpy as np
from sklearn.datasets import make_regression
from sklearn.tree import DecisionTreeRegressor

# ==============================================
# CREATE DUMMY REGRESSION DATA
# ==============================================
X, y = make_regression(n_samples=20, n_features=1, noise=10, random_state=42)

n_samples = len(y)

# ==============================================
# STEP 1: INITIALIZE WEIGHTS
# ==============================================
weights = np.ones(n_samples) / n_samples
print("\nSTEP 1: Initial Weights:")
print(weights)

# PARAMETERS
n_estimators = 5
learning_rate = 1.0

models = []
alphas = []

# ==============================================
# BOOSTING LOOP (AdaBoost.R2 logic)
# ==============================================
for t in range(n_estimators):
    print(f"\n================ ITERATION {t+1} ================")

    # ==============================================
    # STEP 2: TRAIN WEAK LEARNER (REGRESSOR)
    # ==============================================
    stump = DecisionTreeRegressor(max_depth=2)
    stump.fit(X, y, sample_weight=weights)

    predictions = stump.predict(X)
    print("Predictions:", predictions)

    # ==============================================
    # STEP 3: COMPUTE ERROR (REGRESSION VERSION)
    # ==============================================
    # Absolute error
    errors = np.abs(y - predictions)

    # Normalize errors (important in AdaBoost.R2)
    max_error = np.max(errors)
    if max_error == 0:
        max_error = 1e-10

    normalized_error = errors / max_error

    # Weighted error
    error = np.sum(weights * normalized_error)
    print("Weighted Error:", error)

    error = min(max(error, 1e-10), 0.999)
  #Clip error
    # ==============================================
    # STEP 4: COMPUTE MODEL WEIGHT (ALPHA)
    # ==============================================
    beta = error / (1 - error)
    alpha = learning_rate * np.log(1 / beta)

    print("Alpha (Model Weight):", alpha)

    # ==============================================
    # STEP 5: UPDATE WEIGHTS
    # ==============================================
    weights = weights * (beta ** (1 - normalized_error))

    # Normalize
    weights = weights / np.sum(weights)

#KEY DIFFERENCES (REGRESSION vs CLASSIFICATION):
#In classification → discrete outcomes → exponential update works
#In regression → continuous error → need gradual weighting




STEP 1: Initial Weights:
[0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05]

================ ITERATION 1 ================
Predictions: [ 59.75814287  16.42457869 -16.71345693 -69.89874775 -16.71345693
 -69.89874775 -16.71345693 -69.89874775 -16.71345693  16.42457869
 -16.71345693  59.75814287  59.75814287 -16.71345693  59.75814287
 -69.89874775  16.42457869 -16.71345693 -16.71345693  16.42457869]
Weighted Error: 0.4980727521664278
Alpha (Model Weight): 0.007709029512475299

================ ITERATION 2 ================
Predictions: [ 59.75486626  16.42089431 -16.71063795 -69.89873545 -16.71063795
 -69.89873545 -16.71063795 -69.89873545 -16.71063795  16.42089431
 -16.71063795  59.75486626  59.75486626 -16.71063795  59.75486626
 -69.89873545  16.42089431 -16.71063795 -16.71063795  16.42089431]
Weighted Error: 0.49887861437108194
Alpha (Model Weight): 0.004485550036488763

================ ITERATION 3 ================
Predictions: [ 59

In [4]:
def adaboost_regression_predict(X, models, alphas):
    final = np.zeros(X.shape[0])
    total_alpha = np.sum(alphas)

    for alpha, model in zip(alphas, models):
        pred = model.predict(X)
        final += alpha * pred

    return final / total_alpha

In [5]:
final_predictions = adaboost_regression_predict(X, models, alphas)

In [6]:
print("Final Predictions:", final_predictions)
print("Actual Values:", y)

Final Predictions: [-0.52866315 -0.21374766 -0.21374766  0.52866315  0.56770419 -0.56770419
  0.56770419  0.25758919  0.09636735  0.56770419 -0.56770419  0.52866315
  0.56770419 -0.56770419 -0.52866315  0.52866315 -0.52866315 -0.52866315
  0.25758919 -1.        ]
Actual Values: [-1 -1 -1  1  1 -1  1  1 -1  1 -1  1  1 -1 -1  1 -1 -1  1 -1]


### **Bagging **

In [12]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification

# ==============================================
# STEP 0: CREATE DUMMY DATA
# ==============================================
X, y = make_classification(n_samples=10, n_features=5, random_state=42)

print("X:\n", X)
print("y:\n", y)

# ==============================================
# PARAMETERS
# ==============================================
n_estimators = 3   # number of models

models = []

# ==============================================
# STEP 1: BAGGING LOOP
# ==============================================
for i in range(n_estimators):
    print(f"\n=========== MODEL {i+1} ===========")

    # ------------------------------------------
    # STEP 2: BOOTSTRAP SAMPLING
    # ------------------------------------------
    n_samples = X.shape[0]   #number of rows (data points)

    # sample with replacement
    indices = np.random.choice(n_samples, n_samples, replace=True)  #Pick random numbers from 0 to n_samples-1, Pick random numbers from 0 to n_samples-1

    X_sample = X[indices]
    y_sample = y[indices]

    print("Sample indices:", indices)
    print("X_sample:\n", X_sample)
    print("y_sample:\n", y_sample)

    # ------------------------------------------
    # STEP 3: TRAIN MODEL
    # ------------------------------------------
    model = DecisionTreeClassifier(max_depth=2)
    model.fit(X_sample, y_sample)

   s model.append(model)

# ==============================================
# STEP 4: FINAL PREDICTION (MAJORITY VOTING)
# ==============================================
def bagging_predict(X, models):
    all_preds = []    # shape = (n_models,n_samples)

    for model in models:
        pred = model.predict(X)
        print("\nModel prediction:", pred)
        all_preds.append(pred)

    all_preds = np.array(all_preds)

    print("\nAll Predictions:\n", all_preds)

    # Majority voting
    final_pred = []

    for i in range(X.shape[0]):
        values, counts = np.unique(all_preds[:, i], return_counts=True)
        final_pred.append(values[np.argmax(counts)])

    return np.array(final_pred)

# ==============================================
# STEP 5: FINAL OUTPUT
# ==============================================
final_predictions = bagging_predict(X, models)

print("\n===== FINAL RESULT =====")
print("Final Predictions:", final_predictions)
print("Actual Labels:", y)

X:
 [[-0.46171143 -0.58723065 -1.22084365  2.03731034 -1.97171753]
 [ 1.54711661  1.89969252 -0.3011037  -2.39981377  0.83444483]
 [ 0.88394335  1.06833894 -1.32818605 -0.26156038 -0.97007347]
 [ 1.44044386  1.77736657 -0.11564828 -2.79776041  1.51157598]
 [ 1.42544359  1.72725924  0.19686124 -0.71206911 -1.18582677]
 [-0.92533575 -1.14021544  0.2088636   1.69585767 -0.83879234]
 [-1.59731763 -1.96287438  0.73846658  2.57793983 -0.99225135]
 [-0.76281464 -0.9382051  -1.95967012  1.284181   -0.54304815]
 [-0.58067464 -0.72063436  0.82254491  1.39720576 -0.96059253]
 [-2.38935955 -2.8953973   0.17136828  1.20190367  1.97686236]]
y:
 [0 1 1 1 1 0 0 1 0 0]

=========== MODEL 1 ===========
Sample indices: [1 1 8 6 4 2 9 3 8 4]
X_sample:
 [[ 1.54711661  1.89969252 -0.3011037  -2.39981377  0.83444483]
 [ 1.54711661  1.89969252 -0.3011037  -2.39981377  0.83444483]
 [-0.58067464 -0.72063436  0.82254491  1.39720576 -0.96059253]
 [-1.59731763 -1.96287438  0.73846658  2.57793983 -0.99225135]
 [ 1.